# ChronoWitness · E2 en Colab CPU con custodia en Drive

Ejecuta el reajuste E1 y la evaluación retrospectiva E2-A con datos biológicos publicados, código fijado por commit y comprobaciones SHA-256. Los resultados se guardan y releen desde Drive tras E1 y al cerrar E2. No contiene manuscritos.

En **Entorno de ejecución → Cambiar tipo de entorno**, seleccionar **CPU / sin acelerador**. No hace falta GPU. La RAM y la tarifa reales se consultan en el panel Recursos; este cuaderno no afirma una tarifa ni un saldo. La conexión inicial a Drive puede requerir tu autorización de Google. Elegir Ejecutar todas y mantener la sesión atendida conforme al plan de Colab.

No se atribuyen propiedades de validación ciega a E2: los resultados E1 de estos tres experimentos ya fueron examinados. Los ajustes tempranos se evalúan en ventanas posteriores sin solapamiento temporal; las células pueden repetirse entre ambas fases. La calibración predictiva no es una cota física certificada de sesgo para FD7.

Fuente: Lugagne, Blassick y Dunlop, Nature Communications 15, 2148 (2024), https://doi.org/10.1038/s41467-024-46361-1. Datos: https://doi.org/10.5281/zenodo.8114649, CC-BY-4.0. Código ChronoWitness: MIT. No se generan datos científicos sintéticos.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import datetime, hashlib, json, os, shutil, subprocess, sys, uuid, urllib.request
assert (3,10) <= sys.version_info[:2] <= (3,13), 'Esta versión requiere Python 3.10–3.13 para las dependencias fijadas.'
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True)
    assert gpu.returncode != 0 or not gpu.stdout.strip(), 'Cambiar a CPU/sin acelerador antes de ejecutar.'
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
WORK = Path('/content')/('ChronoWitness_E2_'+RUN_ID)
WORK.mkdir(exist_ok=False)
DEST = Path('/content/drive/MyDrive/ChronoWitness/E2')/RUN_ID
print('RAM asignada:', Path('/proc/meminfo').read_text().splitlines()[0])
print('CPU visibles:', os.cpu_count(), 'Afinidad:', len(os.sched_getaffinity(0)))
print('Destino persistente:', DEST)


In [ ]:
SOURCE_COMMIT='3e5d45817e58f540f65fb196d1c878658eb28be2'
EXPECTED={'benchmark.py': '352e0c6c5ebfc0b3c09575fa3bed8bae908976a0cf8c7ecd79aae4178ffca8a3', 'BENCHMARK_PROTOCOL.json': '6c1f8f3d48d6484ecf08a9a3aba571f3a543275006b83af33e8e2eb18e97e39e', 'reference_e1_metrics.json': 'd57bdc73ffc7dc7d712bd94cce1921671824bc5bc2217a117d54cd6b25c60e08', 'calibration.py': '949bc389f8010e3029dbf6da8402dda5386c14bfc1799be14ccfdad87ba35a0b', 'E2_PROTOCOL.json': 'cf4cd502c786b97f922f3596dbb9d9dfa2f9b37da36f01552b420de93a75cb5c'}
for name, digest in EXPECTED.items():
    url=f'https://raw.githubusercontent.com/FacundoFirmenich/ChronoWitness/{SOURCE_COMMIT}/research/e2-memory-check/{name}'
    with urllib.request.urlopen(url, timeout=120) as response: data=response.read()
    assert hashlib.sha256(data).hexdigest()==digest, name
    (WORK/name).write_bytes(data)
DRIVER='"""Execute unchanged E1/E2 calculations with explicit verified Drive checkpoints."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport resource\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport urllib.request\n\nHERE=Path(__file__).resolve().parent\nDEST=Path(sys.argv[1]).resolve()\nARCHIVE_SHA=\'03a788ddeaa08d3b9ef1511208c8c37263cc5bd55af7e9b41f4f3068ab5ab529\'\n\ndef sha(path):\n    h=hashlib.sha256()\n    with path.open(\'rb\') as f:\n        while b:=f.read(1024*1024):h.update(b)\n    return h.hexdigest()\n\ndef copy_verified(source,target):\n    target.parent.mkdir(parents=True,exist_ok=True)\n    expected=sha(source)\n    if target.exists():\n        assert sha(target)==expected, f\'Existing destination differs: {target}\'\n        return\n    # Each run has a unique destination; do not overwrite another run\'s results.\n    with source.open(\'rb\') as inp,target.open(\'xb\') as out:\n        shutil.copyfileobj(inp,out,1024*1024)\n    assert sha(target)==expected, f\'Drive readback mismatch: {target}\'\n\ndef checkpoint(label):\n    target=DEST/label\n    manifest={}\n    for directory in [\'benchmark_results\',\'e2_results\']:\n        folder=HERE/directory\n        if folder.exists():\n            for f in sorted(folder.rglob(\'*\')):\n                if f.is_file():\n                    rel=f.relative_to(HERE)\n                    copy_verified(f,target/rel);manifest[str(rel)]=sha(f)\n    for f in sorted(HERE.glob(\'*.py\'))+sorted(HERE.glob(\'*PROTOCOL.json\')):\n        copy_verified(f,target/\'source\'/f.name);manifest[\'source/\'+f.name]=sha(f)\n    for name in [\'SOURCE_MANIFEST.json\',\'COLAB_RESULT.json\']:\n        f=HERE/name\n        if f.exists():copy_verified(f,target/name);manifest[name]=sha(f)\n    (target/\'SHA256_READBACK.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    print(f\'CHECKPOINT_VERIFIED {target}\',flush=True)\n\ndef main():\n    assert str(DEST).startswith(\'/content/drive/MyDrive/ChronoWitness/E2/\'), DEST\n    DEST.mkdir(parents=True,exist_ok=False)\n    report=dict(status=\'RUNNING\',platform=\'Google Colab CPU\',stages=[],\n        source_commit=\'3e5d45817e58f540f65fb196d1c878658eb28be2\',\n        memory_enforcement=\'No synthetic cgroup cap in Colab; records process RSS and assigned memory.\',\n        assigned_meminfo=Path(\'/proc/meminfo\').read_text().splitlines()[:3],\n        cpu_count=os.cpu_count(),cpu_affinity=sorted(os.sched_getaffinity(0)),\n        python=sys.version,started_unix=time.time())\n    begin=time.monotonic()\n    try:\n        import numpy as np\n        import benchmark as e1\n        # Data persist under the unique run path; future runs can reuse an explicitly supplied verified archive.\n        archive=HERE/\'datasets.zip\'\n        reuse=os.environ.get(\'CHRONOWITNESS_VERIFIED_ARCHIVE\')\n        t=time.monotonic()\n        if reuse:\n            source=Path(reuse);assert sha(source)==ARCHIVE_SHA\n            copy_verified(source,archive)\n        elif not archive.exists():\n            assert shutil.disk_usage(HERE).free>2_000_000_000\n            with urllib.request.urlopen(\'https://zenodo.org/api/records/8114649/files/datasets.zip/content\',timeout=120) as r,archive.open(\'xb\') as f:\n                shutil.copyfileobj(r,f,1024*1024)\n        assert archive.stat().st_size==357636847 and sha(archive)==ARCHIVE_SHA\n        copy_verified(archive,DEST/\'source_data/datasets.zip\')\n        report[\'stages\'].append(dict(stage=\'download_verify_and_archive\',seconds=time.monotonic()-t))\n        # Run the original implementation in a child so its memory is released before E2.\n        code="import benchmark as b; from pathlib import Path; b.download=lambda:(Path(\'datasets.zip\'),\'"+ARCHIVE_SHA+"\'); b.main()"\n        t=time.monotonic();subprocess.run([sys.executable,\'-c\',code],cwd=HERE,check=True)\n        expected=json.loads((HERE/\'reference_e1_metrics.json\').read_text())\n        observed=json.loads((HERE/\'benchmark_results/METRICS.json\').read_text())\n        assert len(expected)==len(observed)==210\n        for a,b in zip(expected,observed):\n            for k in [\'offset\',\'horizon\',\'date\',\'split\',\'model\',\'n\',\'predicted_negative\',\'predicted_above_4095\']:assert a[k]==b[k]\n            for k in [\'rmse\',\'mae\',\'bias\']:assert np.isclose(a[k],b[k],rtol=1e-10,atol=1e-8)\n        report[\'stages\'].append(dict(stage=\'E1_refit_and_reference_check\',seconds=time.monotonic()-t))\n        checkpoint(\'01_E1_complete\')\n        t=time.monotonic();subprocess.run([sys.executable,str(HERE/\'calibration.py\'),str(archive)],cwd=HERE,check=True)\n        report[\'stages\'].append(dict(stage=\'E2_calibration\',seconds=time.monotonic()-t))\n        report[\'status\']=\'COMPLETE_E1_E2_COLAB\'\n    except BaseException:\n        report[\'status\']=\'FAILED_OR_INTERRUPTED\';report[\'error\']=traceback.format_exc();raise\n    finally:\n        report[\'elapsed_seconds\']=time.monotonic()-begin\n        report[\'maximum_child_rss_kib\']=resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss\n        report[\'archive_sha256\']=ARCHIVE_SHA\n        (HERE/\'COLAB_RESULT.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n        checkpoint(\'02_final_or_failure\')\n        print(json.dumps(report,indent=2),flush=True)\n\nif __name__==\'__main__\': main()\n'
(WORK/'run_colab.py').write_text(DRIVER)
(WORK/'SOURCE_MANIFEST.json').write_text(json.dumps(dict(commit=SOURCE_COMMIT, sha256=EXPECTED,
    driver_sha256=hashlib.sha256(DRIVER.encode()).hexdigest()),indent=2))
VENV=WORK/'venv'
subprocess.run([sys.executable,'-m','venv',str(VENV)],check=True)
PY=VENV/'bin/python'
subprocess.run([str(PY),'-m','pip','install','--no-cache-dir','numpy==2.2.6','scipy==1.15.3'],check=True)
print('Código verificado y entorno aislado preparado.')


## Ejecutar y conservar

Se descarga el archivo experimental de 358 MB, se verifica y se copia a Drive. El cálculo se realiza en el disco temporal del runtime para evitar miles de pequeñas operaciones remotas. Para reutilizar un archivo de una ejecución anterior, se puede establecer `CHRONOWITNESS_VERIFIED_ARCHIVE` en la siguiente celda con la ruta real de aquel archivo; su SHA-256 se verificará antes de usarlo. No se borran ni sobrescriben originales.

El cierre incluye métricas completas, parámetros, controles temporales, duración y RSS máximo de proceso. Una desconexión brusca puede impedir el último guardado; el checkpoint E1 ya cerrado permanece. El cuaderno no evita los límites de sesión de Colab.


In [ ]:
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', PYTHONUNBUFFERED='1', GITHUB_SHA=SOURCE_COMMIT)
log=WORK/'COLAB_EXECUTION.log'
try:
    with log.open('w') as out:
        result=subprocess.run([str(PY),str(WORK/'run_colab.py'),str(DEST)],cwd=WORK,env=env,
                              stdout=out,stderr=subprocess.STDOUT,timeout=7200)
finally:
    if DEST.exists():
        target=DEST/'COLAB_EXECUTION.log'
        with log.open('rb') as inp, target.open('xb') as out: shutil.copyfileobj(inp,out)
        assert hashlib.sha256(log.read_bytes()).digest()==hashlib.sha256(target.read_bytes()).digest()
    print(log.read_text()[-16000:])
assert result.returncode==0, 'Ver registro y checkpoints preservados en Drive.'
report=json.loads((WORK/'COLAB_RESULT.json').read_text())
assert report['status']=='COMPLETE_E1_E2_COLAB'
print(json.dumps(report,indent=2))
print('Resultados persistidos y verificados en:', DEST)


## Al terminar

Comprobar los archivos `02_final_or_failure/COLAB_RESULT.json`, `SHA256_READBACK.json` y `e2_results/RUN.json` en Drive. Desconectar el runtime cuando no se vaya a utilizar. Los paquetes instalados y la memoria de Python son temporales; los archivos guardados explícitamente en Drive permanecen allí.
